<a href="https://colab.research.google.com/github/guillemostafa/data/blob/main/Final_DS3_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FINAL

In [ ]:
# Instalación
!pip install gdown
!pip install -q spacy scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 30.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Imports
import pandas as pd
import numpy as np
import re
import gdown
import spacy
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_curve,
    auc
)

np.random.seed(42)
nlp = spacy.load("en_core_web_sm")

In [ ]:
books_rating_id = "1sd_QeCs5l1QmI996H4Cut1X0hLMBXOrs"
books_data_id = "1bLaXNyJWuVrJZviWvWYj3F7ngPfWJIGs"

# Descargar los archivos
gdown.download(f'https://drive.google.com/uc?id={books_rating_id}', 'Books_rating.csv', quiet=False)
gdown.download(f'https://drive.google.com/uc?id={books_data_id}', 'books_data.csv', quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1sd_QeCs5l1QmI996H4Cut1X0hLMBXOrs
From (redirected): https://drive.google.com/uc?id=1sd_QeCs5l1QmI996H4Cut1X0hLMBXOrs&confirm=t&uuid=cc8deaa9-e9d1-4cd6-927a-dbfdc8151584
To: /content/Books_rating.csv
100%|██████████| 2.86G/2.86G [00:41<00:00, 69.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1bLaXNyJWuVrJZviWvWYj3F7ngPfWJIGs
From (redirected): https://drive.google.com/uc?id=1bLaXNyJWuVrJZviWvWYj3F7ngPfWJIGs&confirm=t&uuid=bd6a4fb2-d906-4e04-9af4-03dc9b7ea7ad
To: /content/books_data.csv
100%|██████████| 181M/181M [00:06<00:00, 27.6MB/s]


'books_data.csv'

In [ ]:
df_reviews = pd.read_csv('Books_rating.csv')
# Convertir timestamp a fecha
df_reviews['review/time'] = pd.to_datetime(df_reviews['review/time'], unit='s', errors='coerce')
df_reviews['review/time'] = df_reviews['review/time'].dt.strftime('%d/%m/%Y')

df_reviews.head(5)
# Vista inicial
print("df_reviews tiene {0} registros y {1} columnas.".format(df_reviews.shape[0], df_reviews.shape[1]))

df_reviews tiene 3000000 registros y 10 columnas.


In [ ]:
# Separar helpfulness en votos útiles y totales
df_reviews[['helpful_votes', 'total_votes']] = df_reviews['review/helpfulness'].str.split('/', expand=True).astype(float)
# Calcula proporción de votos útiles
df_reviews['helpfulness_ratio'] = df_reviews['helpful_votes'] / df_reviews['total_votes']
df_reviews.head(5)

In [ ]:
df_books = pd.read_csv('books_data.csv')

In [ ]:
print("df_books tiene {0} registros y {1} columnas.".format(df_books.shape[0], df_books.shape[1]))

In [ ]:
print("Dataset de Libros (cabecera):")
display(df_books.head())

In [ ]:
duplicados_titulo = df_books[df_books.duplicated(subset='Title', keep=False)]
print(f"Registros con títulos duplicados: {len(duplicados_titulo)}")
print(f"Títulos únicos con duplicados: {duplicados_titulo['Title'].nunique()}")

In [ ]:
# Normalizar títulos (evita problemas de espacios o mayúsculas)
df_books['Title'] = df_books['Title'].str.strip().str.lower()
df_reviews['Title'] = df_reviews['Title'].str.strip().str.lower()

In [ ]:
#uno lo dos df por titulo
df_merged = pd.merge(df_reviews, df_books, on='Title', how='left')
df_merged.head(5)


In [ ]:
#Ya tengo todo unificado borro los demas df
del df_books, df_reviews
import gc
gc.collect()